Cell 1：设置 LSRG 零训练验证环境

In [1]:
from pathlib import Path

import numpy as np

from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

print("LSRG feasibility environment ready.")

LSRG feasibility environment ready.


Cell 2：定义局部空间可靠性门控 LSRG

In [2]:
from sklearn.neighbors import NearestNeighbors


def build_symmetric_spatial_neighbors(coords, k=3):
    """
    与当前空间图思路保持一致：
    - kNN
    - exclude self
    - symmetrize
    """
    coords = np.asarray(coords, dtype=np.float32)

    nbrs = NearestNeighbors(
        n_neighbors=k + 1,
        metric="euclidean",
    )
    nbrs.fit(coords)

    indices = nbrs.kneighbors(
        coords,
        return_distance=False,
    )[:, 1:]

    n = coords.shape[0]

    neighbor_sets = [set() for _ in range(n)]

    for i in range(n):
        for j in indices[i]:
            j = int(j)

            neighbor_sets[i].add(j)
            neighbor_sets[j].add(i)

    return neighbor_sets


def row_l2_normalize(x, eps=1e-12):
    norm = np.linalg.norm(
        x,
        axis=1,
        keepdims=True,
    )

    return x / np.maximum(norm, eps)


def compute_local_reliability(
    z_views,
    neighbor_sets,
):
    """
    r_i^v = mean cosine similarity between spot i
    and its spatial neighbors under view v.
    """

    reliabilities = []

    for z in z_views:

        z_norm = row_l2_normalize(z)

        r = np.zeros(
            z.shape[0],
            dtype=np.float32,
        )

        for i, neighbors in enumerate(neighbor_sets):

            if not neighbors:
                r[i] = 0.0
                continue

            idx = np.fromiter(
                neighbors,
                dtype=np.int64,
            )

            sims = (
                z_norm[idx]
                @ z_norm[i]
            )

            r[i] = sims.mean()

        reliabilities.append(r)

    # N x V
    return np.stack(
        reliabilities,
        axis=1,
    )


def lsrg_embedding(
    z_concat,
    coords,
    global_weights,
    n_views=4,
    beta=1.0,
    k=3,
    eps=1e-12,
):
    """
    Zero-training LSRG:

        alpha_i^v =
        softmax(log(w_v) + beta * normalized(r_i^v))

    final embedding:
        concat(sqrt(alpha_i^v) * Z_v)
    """

    z_concat = np.asarray(
        z_concat,
        dtype=np.float32,
    )

    coords = np.asarray(
        coords,
        dtype=np.float32,
    )

    global_weights = np.asarray(
        global_weights,
        dtype=np.float64,
    ).reshape(-1)

    assert len(global_weights) == n_views

    dims = z_concat.shape[1]

    assert dims % n_views == 0

    view_dim = dims // n_views

    z_views = [
        z_concat[
            :,
            v * view_dim:(v + 1) * view_dim,
        ]
        for v in range(n_views)
    ]

    neighbor_sets = (
        build_symmetric_spatial_neighbors(
            coords,
            k=k,
        )
    )

    reliability = (
        compute_local_reliability(
            z_views,
            neighbor_sets,
        )
    )

    # 在每个 spot 的 view 维度上标准化 reliability
    mean = reliability.mean(
        axis=1,
        keepdims=True,
    )

    std = reliability.std(
        axis=1,
        keepdims=True,
    )

    reliability_norm = (
        reliability - mean
    ) / np.maximum(std, 1e-6)

    logits = (
        np.log(
            np.maximum(
                global_weights,
                eps,
            )
        )[None, :]
        +
        beta * reliability_norm
    )

    logits = logits - logits.max(
        axis=1,
        keepdims=True,
    )

    alpha = np.exp(logits)

    alpha = alpha / alpha.sum(
        axis=1,
        keepdims=True,
    )

    weighted_views = []

    for v, z in enumerate(z_views):

        weight = np.sqrt(
            alpha[:, v:v + 1]
        )

        weighted_views.append(
            weight * z
        )

    z_lsrg = np.concatenate(
        weighted_views,
        axis=1,
    )

    return (
        z_lsrg,
        alpha,
        reliability,
    )


print("LSRG functions ready.")

LSRG functions ready.


Cell 3：定义冻结后的正式 KMeans 评价协议

In [3]:
def evaluate_embedding(
    z,
    gt,
    n_clusters,
):
    pred = KMeans(
        n_clusters=n_clusters,
        n_init=20,
        random_state=0,
    ).fit_predict(z)

    ari = adjusted_rand_score(
        gt,
        pred,
    )

    nmi = normalized_mutual_info_score(
        gt,
        pred,
        average_method="max",
    )

    return pred, ari, nmi


print("Evaluation protocol:")
print("KMeans n_init = 20")
print("KMeans random_state = 0")
print("NMI average_method = max")

Evaluation protocol:
KMeans n_init = 20
KMeans random_state = 0
NMI average_method = max


Cell 4：按结果文件自动定位 E18.5_clean 数据目录

In [5]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

print("=" * 80)
print("KAGGLE INPUT DIRECTORIES")
print("=" * 80)

for p in sorted(INPUT_ROOT.iterdir()):
    print(p)

print()
print("=" * 80)
print("SEARCHING FOR z_concat.npy")
print("=" * 80)

z_files = list(INPUT_ROOT.rglob("z_concat.npy"))

if len(z_files) == 0:
    print("没有找到 z_concat.npy")
    print()
    print("下面列出 /kaggle/input 下前两层文件，便于确认数据结构：")

    for dataset_dir in sorted(INPUT_ROOT.iterdir()):
        if not dataset_dir.is_dir():
            continue

        print()
        print(f"[DATASET] {dataset_dir}")

        count = 0
        for p in dataset_dir.rglob("*"):
            if p.is_file():
                print("   ", p.relative_to(dataset_dir))
                count += 1

                if count >= 50:
                    print("    ... only showing first 50 files")
                    break

    raise RuntimeError(
        "当前 /kaggle/input 中没有发现 z_concat.npy，"
        "请把上面的目录输出发给我。"
    )

print(f"Found {len(z_files)} z_concat.npy file(s):")

for i, p in enumerate(z_files):
    print(f"[{i}] {p}")

print()
print("=" * 80)

# 同时检查每个候选目录是否包含我们需要的文件
required_files = [
    "z_concat.npy",
    "coords.npy",
    "gt_labels.npy",
    "sc_weights.npy",
]

valid_dirs = []

for z_path in z_files:
    candidate = z_path.parent

    status = {
        name: (candidate / name).exists()
        for name in required_files
    }

    print()
    print("Candidate:")
    print(candidate)

    for name, exists in status.items():
        print(f"  {name:<20} : {exists}")

    if all(status.values()):
        valid_dirs.append(candidate)

print()
print("=" * 80)
print("VALID RESULT DIRECTORIES")
print("=" * 80)

for i, p in enumerate(valid_dirs):
    print(f"[{i}] {p}")

if len(valid_dirs) == 1:
    RESULT_DIR = valid_dirs[0]

    print()
    print("Auto-selected RESULT_DIR:")
    print(RESULT_DIR)

elif len(valid_dirs) > 1:
    print()
    print("找到多个有效结果目录，暂时不自动选择。")

else:
    print()
    print("没有找到同时包含全部所需文件的目录。")

KAGGLE INPUT DIRECTORIES
/kaggle/input/datasets

SEARCHING FOR z_concat.npy
Found 2 z_concat.npy file(s):
[0] /kaggle/input/datasets/wuvdji/e18-5-clean/e185_clean_200/z_concat.npy
[1] /kaggle/input/datasets/wuvdji/e18-5-clean/e185_clean_smoke/z_concat.npy


Candidate:
/kaggle/input/datasets/wuvdji/e18-5-clean/e185_clean_200
  z_concat.npy         : True
  coords.npy           : True
  gt_labels.npy        : True
  sc_weights.npy       : True

Candidate:
/kaggle/input/datasets/wuvdji/e18-5-clean/e185_clean_smoke
  z_concat.npy         : True
  coords.npy           : True
  gt_labels.npy        : True
  sc_weights.npy       : True

VALID RESULT DIRECTORIES
[0] /kaggle/input/datasets/wuvdji/e18-5-clean/e185_clean_200
[1] /kaggle/input/datasets/wuvdji/e18-5-clean/e185_clean_smoke

找到多个有效结果目录，暂时不自动选择。


Cell 5：在 E18.5 的 50轮和200轮结果上验证 LSRG

In [6]:
from pathlib import Path
import numpy as np

RUN_DIRS = {
    "E18.5-50": Path(
        "/kaggle/input/datasets/wuvdji/e18-5-clean/"
        "e185_clean_smoke"
    ),
    "E18.5-200": Path(
        "/kaggle/input/datasets/wuvdji/e18-5-clean/"
        "e185_clean_200"
    ),
}

N_CLUSTERS = 14
BETA = 1.0
SPATIAL_K = 3

all_results = {}

for run_name, result_dir in RUN_DIRS.items():

    print()
    print("=" * 78)
    print(run_name)
    print("=" * 78)

    z_concat = np.load(
        result_dir / "z_concat.npy"
    )

    coords = np.load(
        result_dir / "coords.npy"
    )

    gt = np.load(
        result_dir / "gt_labels.npy"
    )

    wd_weights = np.load(
        result_dir / "sc_weights.npy"
    )

    print("z_concat shape :", z_concat.shape)
    print("coords shape   :", coords.shape)
    print("GT shape       :", gt.shape)
    print("WD weights     :", wd_weights)

    # --------------------------------------------------
    # 1. 原始冻结 readout
    # --------------------------------------------------

    pred_base, base_ari, base_nmi = (
        evaluate_embedding(
            z_concat,
            gt,
            N_CLUSTERS,
        )
    )

    # --------------------------------------------------
    # 2. LSRG
    # --------------------------------------------------

    z_lsrg, alpha, reliability = (
        lsrg_embedding(
            z_concat=z_concat,
            coords=coords,
            global_weights=wd_weights,
            n_views=4,
            beta=BETA,
            k=SPATIAL_K,
        )
    )

    pred_lsrg, lsrg_ari, lsrg_nmi = (
        evaluate_embedding(
            z_lsrg,
            gt,
            N_CLUSTERS,
        )
    )

    # --------------------------------------------------
    # 3. 输出结果
    # --------------------------------------------------

    print()
    print("Baseline concat-Z")
    print("-----------------")
    print(f"ARI = {base_ari:.6f}")
    print(f"NMI = {base_nmi:.6f}")

    print()
    print(f"LSRG, beta = {BETA}")
    print("-----------------")
    print(f"ARI = {lsrg_ari:.6f}")
    print(f"NMI = {lsrg_nmi:.6f}")

    print()
    print("Change")
    print("------")
    print(
        f"ARI change = "
        f"{lsrg_ari - base_ari:+.6f}"
    )
    print(
        f"NMI change = "
        f"{lsrg_nmi - base_nmi:+.6f}"
    )

    print()
    print("Spot-specific alpha summary")
    print("---------------------------")

    print(
        "mean =",
        np.round(
            alpha.mean(axis=0),
            6,
        ),
    )

    print(
        "std  =",
        np.round(
            alpha.std(axis=0),
            6,
        ),
    )

    print(
        "min  =",
        np.round(
            alpha.min(axis=0),
            6,
        ),
    )

    print(
        "max  =",
        np.round(
            alpha.max(axis=0),
            6,
        ),
    )

    all_results[run_name] = {
        "baseline_ari": base_ari,
        "baseline_nmi": base_nmi,
        "lsrg_ari": lsrg_ari,
        "lsrg_nmi": lsrg_nmi,
        "delta_ari": lsrg_ari - base_ari,
        "delta_nmi": lsrg_nmi - base_nmi,
    }


print()
print("=" * 78)
print("SUMMARY")
print("=" * 78)

for run_name, r in all_results.items():

    print()
    print(run_name)

    print(
        f"ARI: "
        f"{r['baseline_ari']:.6f} "
        f"-> "
        f"{r['lsrg_ari']:.6f} "
        f"({r['delta_ari']:+.6f})"
    )

    print(
        f"NMI: "
        f"{r['baseline_nmi']:.6f} "
        f"-> "
        f"{r['lsrg_nmi']:.6f} "
        f"({r['delta_nmi']:+.6f})"
    )


E18.5-50
z_concat shape : (2129, 512)
coords shape   : (2129, 2)
GT shape       : (2129,)
WD weights     : [0.2501639  0.24958217 0.24989605 0.25035784]

Baseline concat-Z
-----------------
ARI = 0.441822
NMI = 0.532442

LSRG, beta = 1.0
-----------------
ARI = 0.292604
NMI = 0.435438

Change
------
ARI change = -0.149218
NMI change = -0.097004

Spot-specific alpha summary
---------------------------
mean = [0.30396  0.106662 0.394199 0.195179]
std  = [0.189567 0.122147 0.20436  0.150445]
min  = [0.032172 0.031981 0.032053 0.032132]
max  = [0.770272 0.767057 0.769953 0.760369]

E18.5-200
z_concat shape : (2129, 512)
coords shape   : (2129, 2)
GT shape       : (2129,)
WD weights     : [0.25013176 0.25013858 0.24984518 0.24988447]

Baseline concat-Z
-----------------
ARI = 0.328863
NMI = 0.533971

LSRG, beta = 1.0
-----------------
ARI = 0.300416
NMI = 0.497161

Change
------
ARI change = -0.028447
NMI change = -0.036810

Spot-specific alpha summary
---------------------------
mean = [0

Cell 6：测试边界保护的空间残差 refinement

In [7]:
import numpy as np
from pathlib import Path
from sklearn.neighbors import NearestNeighbors


def boundary_aware_refinement(
    z,
    coords,
    k=3,
    eps=1e-12,
):
    """
    Parameter-free feasibility version.

    Spatial affinity:
        exp(-d_s^2 / (2 sigma_s^2))

    Latent affinity:
        exp(-d_z^2 / (2 sigma_z^2))

    sigma_s and sigma_z are determined by
    median edge distances.

    Residual refinement:
        z'_i = (z_i + c_i * neighbor_i) / (1 + c_i)

    where c_i is the mean local affinity.
    """

    z = np.asarray(z, dtype=np.float32)
    coords = np.asarray(coords, dtype=np.float32)

    n = z.shape[0]

    # ---------------------------------------------
    # 1. Build the same kNN spatial neighborhood
    # ---------------------------------------------

    nn = NearestNeighbors(
        n_neighbors=k + 1,
        metric="euclidean",
    )

    nn.fit(coords)

    indices = nn.kneighbors(
        coords,
        return_distance=False,
    )[:, 1:]

    # Symmetrize
    neighbor_sets = [set() for _ in range(n)]

    for i in range(n):
        for j in indices[i]:
            j = int(j)

            neighbor_sets[i].add(j)
            neighbor_sets[j].add(i)

    # ---------------------------------------------
    # 2. Normalize Z only for latent-distance
    #    calculation, not for final representation
    # ---------------------------------------------

    z_norm = z / np.maximum(
        np.linalg.norm(
            z,
            axis=1,
            keepdims=True,
        ),
        eps,
    )

    spatial_distances = []
    latent_distances = []

    edges = []

    for i in range(n):
        for j in neighbor_sets[i]:

            # only collect each undirected edge once
            if j <= i:
                continue

            ds = np.linalg.norm(
                coords[i] - coords[j]
            )

            dz = np.linalg.norm(
                z_norm[i] - z_norm[j]
            )

            edges.append((i, j, ds, dz))

            if ds > 0:
                spatial_distances.append(ds)

            if dz > 0:
                latent_distances.append(dz)

    spatial_distances = np.asarray(
        spatial_distances,
        dtype=np.float64,
    )

    latent_distances = np.asarray(
        latent_distances,
        dtype=np.float64,
    )

    sigma_s = float(
        np.median(spatial_distances)
    )

    sigma_z = float(
        np.median(latent_distances)
    )

    print(f"sigma_spatial = {sigma_s:.6f}")
    print(f"sigma_latent  = {sigma_z:.6f}")

    assert sigma_s > 0
    assert sigma_z > 0

    # ---------------------------------------------
    # 3. Calculate symmetric edge affinities
    # ---------------------------------------------

    weighted_sum = np.zeros_like(
        z,
        dtype=np.float64,
    )

    affinity_sum = np.zeros(
        n,
        dtype=np.float64,
    )

    degree = np.zeros(
        n,
        dtype=np.float64,
    )

    for i, j, ds, dz in edges:

        spatial_affinity = np.exp(
            -0.5 * (ds / sigma_s) ** 2
        )

        latent_affinity = np.exp(
            -0.5 * (dz / sigma_z) ** 2
        )

        affinity = (
            spatial_affinity
            * latent_affinity
        )

        weighted_sum[i] += (
            affinity * z[j]
        )

        weighted_sum[j] += (
            affinity * z[i]
        )

        affinity_sum[i] += affinity
        affinity_sum[j] += affinity

        degree[i] += 1
        degree[j] += 1

    # ---------------------------------------------
    # 4. Weighted neighbor representation
    # ---------------------------------------------

    neighbor_rep = (
        weighted_sum
        /
        np.maximum(
            affinity_sum[:, None],
            eps,
        )
    )

    # Local confidence lies approximately in [0, 1]
    confidence = (
        affinity_sum
        /
        np.maximum(
            degree,
            1.0,
        )
    )

    confidence = np.clip(
        confidence,
        0.0,
        1.0,
    )

    # ---------------------------------------------
    # 5. Boundary-preserving residual update
    # ---------------------------------------------

    z_refined = (
        z
        +
        confidence[:, None]
        * neighbor_rep
    ) / (
        1.0
        +
        confidence[:, None]
    )

    return (
        z_refined.astype(np.float32),
        confidence,
    )


RUN_DIRS = {
    "E18.5-50": Path(
        "/kaggle/input/datasets/wuvdji/e18-5-clean/"
        "e185_clean_smoke"
    ),

    "E18.5-200": Path(
        "/kaggle/input/datasets/wuvdji/e18-5-clean/"
        "e185_clean_200"
    ),
}

N_CLUSTERS = 14
SPATIAL_K = 3

results = {}

for run_name, result_dir in RUN_DIRS.items():

    print()
    print("=" * 76)
    print(run_name)
    print("=" * 76)

    z = np.load(
        result_dir / "z_concat.npy"
    )

    coords = np.load(
        result_dir / "coords.npy"
    )

    gt = np.load(
        result_dir / "gt_labels.npy"
    )

    # Original official readout
    _, base_ari, base_nmi = (
        evaluate_embedding(
            z,
            gt,
            N_CLUSTERS,
        )
    )

    # Boundary-aware refinement
    z_refined, confidence = (
        boundary_aware_refinement(
            z,
            coords,
            k=SPATIAL_K,
        )
    )

    _, new_ari, new_nmi = (
        evaluate_embedding(
            z_refined,
            gt,
            N_CLUSTERS,
        )
    )

    print()
    print("Baseline concat-Z")
    print("-----------------")
    print(f"ARI = {base_ari:.6f}")
    print(f"NMI = {base_nmi:.6f}")

    print()
    print("Boundary-aware refinement")
    print("-------------------------")
    print(f"ARI = {new_ari:.6f}")
    print(f"NMI = {new_nmi:.6f}")

    print()
    print("Change")
    print("------")
    print(
        f"ARI change = "
        f"{new_ari - base_ari:+.6f}"
    )

    print(
        f"NMI change = "
        f"{new_nmi - base_nmi:+.6f}"
    )

    print()
    print("Local confidence")
    print("----------------")
    print(
        f"mean = {confidence.mean():.6f}"
    )

    print(
        f"std  = {confidence.std():.6f}"
    )

    print(
        f"min  = {confidence.min():.6f}"
    )

    print(
        f"max  = {confidence.max():.6f}"
    )

    results[run_name] = {
        "base_ari": base_ari,
        "base_nmi": base_nmi,
        "new_ari": new_ari,
        "new_nmi": new_nmi,
    }


print()
print("=" * 76)
print("SUMMARY")
print("=" * 76)

for run_name, r in results.items():

    print()
    print(run_name)

    print(
        f"ARI: "
        f"{r['base_ari']:.6f}"
        f" -> "
        f"{r['new_ari']:.6f}"
        f" "
        f"({r['new_ari'] - r['base_ari']:+.6f})"
    )

    print(
        f"NMI: "
        f"{r['base_nmi']:.6f}"
        f" -> "
        f"{r['new_nmi']:.6f}"
        f" "
        f"({r['new_nmi'] - r['base_nmi']:+.6f})"
    )


E18.5-50
sigma_spatial = 1.000000
sigma_latent  = 0.379222

Baseline concat-Z
-----------------
ARI = 0.441822
NMI = 0.532442

Boundary-aware refinement
-------------------------
ARI = 0.452612
NMI = 0.536439

Change
------
ARI change = +0.010790
NMI change = +0.003996

Local confidence
----------------
mean = 0.355543
std  = 0.090830
min  = 0.068358
max  = 0.593990

E18.5-200
sigma_spatial = 1.000000
sigma_latent  = 0.426014

Baseline concat-Z
-----------------
ARI = 0.328863
NMI = 0.533971

Boundary-aware refinement
-------------------------
ARI = 0.418834
NMI = 0.553210

Change
------
ARI change = +0.089971
NMI change = +0.019239

Local confidence
----------------
mean = 0.354286
std  = 0.086473
min  = 0.086454
max  = 0.589605

SUMMARY

E18.5-50
ARI: 0.441822 -> 0.452612 (+0.010790)
NMI: 0.532442 -> 0.536439 (+0.003996)

E18.5-200
ARI: 0.328863 -> 0.418834 (+0.089971)
NMI: 0.533971 -> 0.553210 (+0.019239)


Cell 7：测试 Mutual Spatial-Embedding Neighbor Refinement（MSER）

In [8]:
import numpy as np
from pathlib import Path
from sklearn.neighbors import NearestNeighbors


def mser_refinement(
    z,
    coords,
    spatial_k=3,
    latent_k=20,
    eps=1e-12,
):
    """
    MSER:
    Mutual Spatial-Embedding Neighbor Refinement

    Trusted edge (i, j) must satisfy:
    1. i and j are connected in the symmetrized spatial kNN graph;
    2. i is in j's latent kNN AND j is in i's latent kNN.

    latent_k=20 is inherited from the current feature-graph setting,
    rather than selected according to ARI/NMI.

    Only trusted neighbors contribute to residual refinement.
    """

    z = np.asarray(z, dtype=np.float32)
    coords = np.asarray(coords, dtype=np.float32)

    n = z.shape[0]

    # --------------------------------------------------
    # 1. Build symmetric spatial kNN graph
    # --------------------------------------------------

    spatial_nn = NearestNeighbors(
        n_neighbors=spatial_k + 1,
        metric="euclidean",
    )

    spatial_nn.fit(coords)

    spatial_idx = spatial_nn.kneighbors(
        coords,
        return_distance=False,
    )[:, 1:]

    spatial_sets = [set() for _ in range(n)]

    for i in range(n):
        for j in spatial_idx[i]:
            j = int(j)

            spatial_sets[i].add(j)
            spatial_sets[j].add(i)

    # --------------------------------------------------
    # 2. L2-normalize latent representation
    # --------------------------------------------------

    z_norm = z / np.maximum(
        np.linalg.norm(
            z,
            axis=1,
            keepdims=True,
        ),
        eps,
    )

    # --------------------------------------------------
    # 3. Build latent kNN graph
    # --------------------------------------------------

    latent_nn = NearestNeighbors(
        n_neighbors=latent_k + 1,
        metric="cosine",
    )

    latent_nn.fit(z_norm)

    latent_idx = latent_nn.kneighbors(
        z_norm,
        return_distance=False,
    )[:, 1:]

    latent_sets = [
        set(map(int, row))
        for row in latent_idx
    ]

    # --------------------------------------------------
    # 4. Select trusted spatial + reciprocal latent edges
    # --------------------------------------------------

    trusted_edges = []
    trusted_sets = [set() for _ in range(n)]

    latent_distances = []

    total_spatial_edges = 0

    for i in range(n):

        for j in spatial_sets[i]:

            if j <= i:
                continue

            total_spatial_edges += 1

            reciprocal_latent = (
                j in latent_sets[i]
                and
                i in latent_sets[j]
            )

            if not reciprocal_latent:
                continue

            dz = np.linalg.norm(
                z_norm[i] - z_norm[j]
            )

            trusted_edges.append(
                (i, j, dz)
            )

            trusted_sets[i].add(j)
            trusted_sets[j].add(i)

            if dz > 0:
                latent_distances.append(dz)

    if len(trusted_edges) == 0:
        raise RuntimeError(
            "MSER found no trusted spatial-latent edges."
        )

    latent_distances = np.asarray(
        latent_distances,
        dtype=np.float64,
    )

    sigma_z = float(
        np.median(latent_distances)
    )

    assert sigma_z > 0

    # --------------------------------------------------
    # 5. Latent-similarity weighted trusted aggregation
    # --------------------------------------------------

    weighted_sum = np.zeros_like(
        z,
        dtype=np.float64,
    )

    affinity_sum = np.zeros(
        n,
        dtype=np.float64,
    )

    spatial_degree = np.array(
        [
            len(spatial_sets[i])
            for i in range(n)
        ],
        dtype=np.float64,
    )

    trusted_degree = np.array(
        [
            len(trusted_sets[i])
            for i in range(n)
        ],
        dtype=np.float64,
    )

    for i, j, dz in trusted_edges:

        affinity = np.exp(
            -0.5 * (dz / sigma_z) ** 2
        )

        weighted_sum[i] += (
            affinity * z[j]
        )

        weighted_sum[j] += (
            affinity * z[i]
        )

        affinity_sum[i] += affinity
        affinity_sum[j] += affinity

    neighbor_rep = (
        weighted_sum
        /
        np.maximum(
            affinity_sum[:, None],
            eps,
        )
    )

    # --------------------------------------------------
    # 6. Conservative confidence
    #
    # Only a fraction of spatial neighbors survive.
    # Confidence therefore stays in [0, 1].
    # --------------------------------------------------

    trusted_ratio = (
        trusted_degree
        /
        np.maximum(
            spatial_degree,
            1.0,
        )
    )

    mean_affinity = (
        affinity_sum
        /
        np.maximum(
            trusted_degree,
            1.0,
        )
    )

    confidence = (
        trusted_ratio
        * mean_affinity
    )

    confidence = np.clip(
        confidence,
        0.0,
        1.0,
    )

    # Spots without trusted neighbors remain unchanged
    has_trusted = (
        trusted_degree > 0
    )

    z_refined = z.astype(
        np.float64
    ).copy()

    z_refined[has_trusted] = (
        z[has_trusted]
        +
        confidence[
            has_trusted,
            None,
        ]
        *
        neighbor_rep[
            has_trusted
        ]
    ) / (
        1.0
        +
        confidence[
            has_trusted,
            None,
        ]
    )

    diagnostics = {
        "sigma_z": sigma_z,
        "total_spatial_edges": total_spatial_edges,
        "trusted_edges": len(trusted_edges),
        "trusted_edge_ratio": (
            len(trusted_edges)
            /
            max(total_spatial_edges, 1)
        ),
        "spots_with_trusted_neighbors": int(
            has_trusted.sum()
        ),
        "spots_without_trusted_neighbors": int(
            (~has_trusted).sum()
        ),
    }

    return (
        z_refined.astype(np.float32),
        confidence,
        diagnostics,
    )


RUN_DIRS = {
    "E18.5-50": Path(
        "/kaggle/input/datasets/wuvdji/e18-5-clean/"
        "e185_clean_smoke"
    ),
    "E18.5-200": Path(
        "/kaggle/input/datasets/wuvdji/e18-5-clean/"
        "e185_clean_200"
    ),
}

N_CLUSTERS = 14
SPATIAL_K = 3
LATENT_K = 20

comparison = {}

for run_name, result_dir in RUN_DIRS.items():

    print()
    print("=" * 78)
    print(run_name)
    print("=" * 78)

    z = np.load(
        result_dir / "z_concat.npy"
    )

    coords = np.load(
        result_dir / "coords.npy"
    )

    gt = np.load(
        result_dir / "gt_labels.npy"
    )

    # --------------------------------------------------
    # Baseline
    # --------------------------------------------------

    _, base_ari, base_nmi = (
        evaluate_embedding(
            z,
            gt,
            N_CLUSTERS,
        )
    )

    # --------------------------------------------------
    # BSRR — use the exact frozen Cell 6 implementation
    # --------------------------------------------------

    z_bsrr, bsrr_conf = (
        boundary_aware_refinement(
            z,
            coords,
            k=SPATIAL_K,
        )
    )

    _, bsrr_ari, bsrr_nmi = (
        evaluate_embedding(
            z_bsrr,
            gt,
            N_CLUSTERS,
        )
    )

    # --------------------------------------------------
    # MSER
    # --------------------------------------------------

    z_mser, mser_conf, diag = (
        mser_refinement(
            z,
            coords,
            spatial_k=SPATIAL_K,
            latent_k=LATENT_K,
        )
    )

    _, mser_ari, mser_nmi = (
        evaluate_embedding(
            z_mser,
            gt,
            N_CLUSTERS,
        )
    )

    print()
    print("MSER diagnostics")
    print("----------------")
    print(
        f"sigma_latent           = "
        f"{diag['sigma_z']:.6f}"
    )
    print(
        f"spatial edges          = "
        f"{diag['total_spatial_edges']}"
    )
    print(
        f"trusted edges          = "
        f"{diag['trusted_edges']}"
    )
    print(
        f"trusted edge ratio     = "
        f"{diag['trusted_edge_ratio']:.6f}"
    )
    print(
        f"spots with neighbors   = "
        f"{diag['spots_with_trusted_neighbors']}"
    )
    print(
        f"spots without neighbors= "
        f"{diag['spots_without_trusted_neighbors']}"
    )

    print()
    print("MSER confidence")
    print("---------------")
    print(
        f"mean = {mser_conf.mean():.6f}"
    )
    print(
        f"std  = {mser_conf.std():.6f}"
    )
    print(
        f"min  = {mser_conf.min():.6f}"
    )
    print(
        f"max  = {mser_conf.max():.6f}"
    )

    print()
    print("Performance")
    print("-----------")
    print(
        f"Baseline | "
        f"ARI={base_ari:.6f} "
        f"NMI={base_nmi:.6f}"
    )

    print(
        f"BSRR     | "
        f"ARI={bsrr_ari:.6f} "
        f"NMI={bsrr_nmi:.6f}"
    )

    print(
        f"MSER     | "
        f"ARI={mser_ari:.6f} "
        f"NMI={mser_nmi:.6f}"
    )

    print()
    print("MSER vs baseline")
    print("----------------")
    print(
        f"ΔARI = "
        f"{mser_ari - base_ari:+.6f}"
    )
    print(
        f"ΔNMI = "
        f"{mser_nmi - base_nmi:+.6f}"
    )

    print()
    print("MSER vs BSRR")
    print("------------")
    print(
        f"ΔARI = "
        f"{mser_ari - bsrr_ari:+.6f}"
    )
    print(
        f"ΔNMI = "
        f"{mser_nmi - bsrr_nmi:+.6f}"
    )

    comparison[run_name] = {
        "baseline_ari": base_ari,
        "baseline_nmi": base_nmi,
        "bsrr_ari": bsrr_ari,
        "bsrr_nmi": bsrr_nmi,
        "mser_ari": mser_ari,
        "mser_nmi": mser_nmi,
    }


print()
print("=" * 78)
print("FINAL BSRR vs MSER SUMMARY")
print("=" * 78)

for run_name, r in comparison.items():

    print()
    print(run_name)

    print(
        f"Baseline : "
        f"{r['baseline_ari']:.6f} / "
        f"{r['baseline_nmi']:.6f}"
    )

    print(
        f"BSRR     : "
        f"{r['bsrr_ari']:.6f} / "
        f"{r['bsrr_nmi']:.6f}"
    )

    print(
        f"MSER     : "
        f"{r['mser_ari']:.6f} / "
        f"{r['mser_nmi']:.6f}"
    )


E18.5-50
sigma_spatial = 1.000000
sigma_latent  = 0.379222

MSER diagnostics
----------------
sigma_latent           = 0.344645
spatial edges          = 3892
trusted edges          = 1689
trusted edge ratio     = 0.433967
spots with neighbors   = 1634
spots without neighbors= 495

MSER confidence
---------------
mean = 0.260981
std  = 0.202300
min  = 0.000000
max  = 0.969148

Performance
-----------
Baseline | ARI=0.441822 NMI=0.532442
BSRR     | ARI=0.452612 NMI=0.536439
MSER     | ARI=0.446741 NMI=0.538342

MSER vs baseline
----------------
ΔARI = +0.004918
ΔNMI = +0.005900

MSER vs BSRR
------------
ΔARI = -0.005872
ΔNMI = +0.001903

E18.5-200
sigma_spatial = 1.000000
sigma_latent  = 0.426014

MSER diagnostics
----------------
sigma_latent           = 0.378757
spatial edges          = 3892
trusted edges          = 1906
trusted edge ratio     = 0.489723
spots with neighbors   = 1772
spots without neighbors= 357

MSER confidence
---------------
mean = 0.297395
std  = 0.203560
min  = 